In [1]:
# Google Drive on Kaggle, as on molab: `fs` (gdrive_fsspec) for the version 21 cell (cells/v21_base.py).
# The first time, in an interactive session (Run, not Commit): open the link it prints, allow Drive, paste the code
# it gives you into the box. It then prints your sign-in: put it in a Kaggle secret named GDRIVE_CREDS (Add-ons ->
# Secrets), and clear this cell's output. From then on (Commit too) it signs in from that secret by itself.
import json, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdrive_fsspec", "pydata_google_auth"], check=False)
from gdrive_fsspec import GoogleDriveFileSystem            # noqa: E402

_saved = None
try:
    from kaggle_secrets import UserSecretsClient
    _saved = UserSecretsClient().get_secret("GDRIVE_CREDS")
except Exception:
    pass
if _saved:                                                 # the sign-in kept in the secret
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    _cred = Credentials.from_authorized_user_info(json.loads(_saved))
    fs = GoogleDriveFileSystem(token="anon", use_listings_cache=False, skip_instance_cache=True)
    fs.srv = build("drive", "v3", credentials=_cred)
    fs.files = fs.srv.files()
else:                                                      # the first time: sign in by hand
    fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True,
                               auth_kwargs={"use_local_webserver": False})
print("Drive's harraf folder:", [p.rsplit("/", 1)[-1] for p in fs.ls("harraf")][:12])
if not _saved:
    print("\nSigned in. Put the line below in a Kaggle secret named GDRIVE_CREDS (Add-ons -> Secrets), then clear "
          "this output (it opens your Drive):\n")
    print(fs.srv._http.credentials.to_json())

Drive's harraf folder: ['bench', 'chat20_data.jsonl.gz', 'harraf2', 'harraf2_base_chat20.zip', 'harraf2_base_n.zip', 'harraf2_kaggle_code.zip', 'hf_token.txt', 'input', 'kaggle.json', 'knowledge_facts.bin', 'new21', 'newdata']


In [2]:
# Harraf version 21, cell 1 (marimo / molab): the base model reads more, and more kinds of, Persian text, grown to
# 32 blocks. Google Drive is reached through `fs` (gdrive_fsspec, the notebook's first cell). Everything runs on this
# machine; the new data and the save to go on from are sent to Drive while it runs, so after a disconnection the
# same cell goes on where it stopped. All inside one function: marimo sees no names that clash with other cells.
# On Kaggle (GPU T4 x2, Internet on) it runs as it is after cells/kaggle_drive.py, which makes `fs` there.

def _harraf_v21(fs):
    import glob, json, os, shutil, signal, subprocess, sys, threading, time, urllib.request, zipfile

    HOURS = 8.5            # the whole cell: making the data and training (the export takes a few minutes more)
    DATA_HOURS = 2.0       # making the new data at most; what is not made by then waits for the next run
    NAME = "harraf2_base_21"
    BRANCH = "claude/zealous-mendel-8bjvg1"
    R = "harraf"           # the folder in Google Drive
    # tokens to make from each source (at most), in this order; the share of every batch each gets
    SOURCES = [            # name, tokens, share, extra flags
        ("youtube", 2.5e8, 0.08, []),               # spoken Persian of YouTube videos (their words only)
        ("fineweb_hq", 5.0e8, 0.26, []),
        ("blog", 1.2e8, 0.04, []),
        ("telegram", 1.0e8, 0.04, []),
        ("summary", 6.0e7, 0.03, []),
        ("stories_fa", 3.0e7, 0.01, []),
        ("naab", 1.5e8, 0.08, ["--reverse"]),    # from its last file back: files version 19 did not read
    ]
    # train.bin's own sources, every batch: less Wikipedia than before (35%), more talk, posts, stories and poems
    BASE = "wiki=0.25,web=0.14,subs=0.12,forum=0.07,tweets=0.06,stories=0.06,poetry=0.08,qa=0.06,dict=0.04,quran=0.02"
    # the cool-down (its last 20%): the best of every kind, not Wikipedia alone; spoken Persian (films' subtitles,
    # YouTube) among it
    COOL = ("wiki=0.20,facts=0.05,fineweb_hq=0.16,subs=0.07,youtube=0.03,forum=0.04,tweets=0.03,blog=0.05,books=0.05,"
            "poetry=0.06,stories=0.04,summary=0.04,telegram=0.03,naab=0.04,qa=0.06,dict=0.03,quran=0.02")
    T0 = time.time()
    WORK = "/marimo/harraf/v21" if os.path.isdir("/marimo") else "/tmp/harraf21"   # molab, or Kaggle and others
    CODE, NEW, PDF, DATA = (os.path.join(WORK, x) for x in ("code", "new21", "pdf", "data"))
    LIGHT = os.path.join(WORK, "v21_light.pt")
    for d in (WORK, NEW, PDF, DATA):
        os.makedirs(d, exist_ok=True)

    # ---- Google Drive through fs ---------------------------------------------------------------------
    def rsize(r):
        try:
            return fs.info(r).get("size")
        except Exception:
            return None

    def pull(r, local):
        """A Drive file to this machine, unless the same size is here already."""
        size = rsize(r)
        if size is None:
            return False
        if os.path.exists(local) and os.path.getsize(local) == int(size):
            return True
        os.makedirs(os.path.dirname(local), exist_ok=True)
        print("from Drive:", r, f"({int(size) / 1e6:,.0f} MB)", flush=True)
        fs.get_file(r, local + ".part")
        os.replace(local + ".part", local)
        return True

    def remove(r):
        """Every Drive file of this name (Drive can keep two of the same name), each by its id."""
        parent = r.rsplit("/", 1)[0]
        try:
            listed = fs.ls(parent, detail=True)
        except FileNotFoundError:
            return False
        for f in listed:
            if f["name"] == r:
                try:
                    fs.files.delete(fileId=f["id"], supportsAllDrives=True).execute()   # gdrive_fsspec's own service
                except AttributeError:
                    fs.rm(r)
        return True

    def push(local, r):
        """This machine's file to Drive, in place of the one there (Drive would keep two of the same name)."""
        if not remove(r):
            fs.makedirs(r.rsplit("/", 1)[0], exist_ok=True)
        fs.put_file(local, r)

    def files_in(r):
        try:
            return [f for f in fs.find(r, detail=True).values() if f.get("type") != "directory"]
        except FileNotFoundError:
            return []

    if rsize(f"{R}/{NAME}.zip") is not None:
        print(f"{R}/{NAME}.zip is in Drive already: version 21's base model is made")
        return

    # ---- the code: from GitHub, else harraf2_kaggle_code.zip in Drive -------------------------------------
    z = os.path.join(WORK, "code.zip")
    try:
        urllib.request.urlretrieve(f"https://github.com/alipyth/newfarsimodel/raw/{BRANCH}/kaggle/harraf2_kaggle_code.zip", z)
        print("code: from GitHub", flush=True)
    except Exception as e:
        assert pull(f"{R}/harraf2_kaggle_code.zip", z), "harraf2_kaggle_code.zip is not in Drive's harraf folder"
        print(f"code: from Drive (GitHub: {type(e).__name__})", flush=True)
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(z).extractall(CODE)
    assert os.path.exists(os.path.join(CODE, "books.py")), "this code zip is older than version 21"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "pyarrow", "pymupdf", "requests",
                    "huggingface_hub", "numpy"], check=False)
    if pull(f"{R}/hf_read_token.txt", os.path.join(WORK, "hf_token.txt")) and not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = open(os.path.join(WORK, "hf_token.txt")).read().strip()

    def sh(*cmd, timeout=None, check=True):
        """A step, its output shown here; stopped (with its helper processes) when out of time."""
        print("\n$", " ".join(str(c) for c in cmd), flush=True)
        p = subprocess.Popen([str(c) for c in cmd], cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, start_new_session=True)
        killed = []
        timer = None
        if timeout:
            def stop():
                killed.append(1)
                os.killpg(p.pid, signal.SIGKILL)
            timer = threading.Timer(timeout, stop)
            timer.start()
        try:
            for line in p.stdout:
                print(line, end="", flush=True)
            rc = p.wait()
        except BaseException:                            # the cell stopped by hand: the step stops with it
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            raise
        if timer:
            timer.cancel()
        if killed:
            print("(stopped: out of time for this step; what it finished is kept)", flush=True)
            return False
        if check and rc != 0:
            raise RuntimeError(f"step failed ({rc}): {' '.join(str(c) for c in cmd)}")
        return rc == 0

    # ---- the model to start from, its data, and what earlier runs of this cell left in Drive --------------
    assert pull(f"{R}/harraf2_base_n.zip", os.path.join(WORK, "harraf2_base_n.zip")), "harraf2_base_n.zip is not in Drive"
    found = glob.glob(os.path.join(WORK, "**", "harraf2_base_n", "model.pt"), recursive=True)
    if not found:
        zipfile.ZipFile(os.path.join(WORK, "harraf2_base_n.zip")).extractall(os.path.join(WORK, "model"))
        found = glob.glob(os.path.join(WORK, "model", "**", "model.pt"), recursive=True)
    SRC = os.path.dirname(found[0])
    print("starting from:", SRC, flush=True)
    every = files_in(R)
    bins = [f["name"] for f in every if f["name"].endswith("/train.bin") and "/new21/" not in f["name"]]
    assert bins, "train.bin (the pretraining data) is not in Drive"
    folder = bins[0].rsplit("/", 1)[0]
    for f in every:
        name = f["name"]
        base = name.rsplit("/", 1)[-1]
        if name.rsplit("/", 1)[0] == folder and (base in ("train.bin", "meta.json", "tokenizer.json", "morph.json")
                                                 or (base.startswith("val") and base.endswith(".txt"))):
            pull(name, os.path.join(DATA, base))
    for f in ("tokenizer.json", "morph.json"):
        if not os.path.exists(os.path.join(DATA, f)):
            shutil.copy(os.path.join(SRC, f), os.path.join(DATA, f))
    assert open(os.path.join(DATA, "tokenizer.json"), "rb").read() == open(os.path.join(SRC, "tokenizer.json"), "rb").read(), \
        "train.bin's tokenizer is not harraf2_base_n's"
    held = [os.path.join(DATA, f) for f in os.listdir(DATA) if f.startswith("val") and f.endswith(".txt")]
    for f in files_in(f"{R}/new21"):                     # the data an earlier run of this cell made
        if not f["name"].endswith((".part", ".tmp")) and "/cache/" not in f["name"]:
            pull(f["name"], os.path.join(NEW, f["name"].split("/new21/", 1)[1]))
    if not os.path.exists(os.path.join(NEW, "hashes.npy")):
        old = [f["name"] for f in every if f["name"].endswith("/hashes.npy")]
        if old:
            pull(old[0], os.path.join(NEW, "hashes.npy"))  # the paragraphs version 19 read: not read again
    for f in files_in(f"{R}/pdf"):                       # your books
        pull(f["name"], os.path.join(PDF, f["name"].split("/pdf/", 1)[1]))
    started = pull(f"{R}/v21_light.pt", LIGHT)            # training began in an earlier run: go on from it
    facts = [f["name"] for f in every if f["name"].endswith("/knowledge_facts.bin")]
    OUT = os.path.join(WORK, "round")
    os.makedirs(os.path.join(OUT, "facts"), exist_ok=True)
    if facts:
        pull(facts[0], os.path.join(OUT, "facts", "facts.bin"))   # the facts of version 19: not made again

    # ---- sending to Drive while it runs: the new data and the save to go on from ---------------------------
    sent, stop, lock = {}, threading.Event(), threading.Lock()

    def send_new():
        todo = [(LIGHT, f"{R}/v21_light.pt")] if os.path.exists(LIGHT) else []
        for p in glob.glob(os.path.join(NEW, "*")):
            if os.path.isfile(p) and not p.endswith((".part", ".tmp", ".tmp.npy")):
                todo.append((p, f"{R}/new21/{os.path.basename(p)}"))
        for local, r in todo:
            try:
                m = (os.path.getmtime(local), os.path.getsize(local))
                if sent.get(local) != m and time.time() - m[0] > 20 and os.path.exists(local):   # finished being written
                    with lock:
                        push(local, r)
                    sent[local] = m
                    print(f"[sent to Drive: {r}]", flush=True)
            except Exception as e:
                print(f"[not sent to Drive yet: {r} ({type(e).__name__}: {str(e)[:100]})]", flush=True)

    for p in glob.glob(os.path.join(NEW, "*")) + [LIGHT]:   # what came from Drive is there already
        if os.path.isfile(p):
            sent[p] = (os.path.getmtime(p), os.path.getsize(p))

    def sender():
        while not stop.wait(300):
            send_new()

    thread = threading.Thread(target=sender, daemon=True)
    thread.start()
    try:
        # ---- the new data: your books, then each source --------------------------------------------------
        data_end = T0 + (0 if started else DATA_HOURS) * 3600
        if started:
            print("training began in an earlier run: going on from v21_light.pt with the data made then", flush=True)
        if time.time() < data_end and os.listdir(PDF):
            sh(sys.executable, "books.py", "--src", PDF, "--out", NEW, "--tok", SRC, "--ref", os.path.join(DATA, "val.txt"))
            if os.path.exists(os.path.join(NEW, "books.json")):
                bfile = json.load(open(os.path.join(NEW, "books.json")))["file"]
                keep = "books_" + bfile[:-4] + "_p2."                     # this making of your books
                for p in glob.glob(os.path.join(NEW, "books_*")):        # an earlier making of your books: replaced
                    b = os.path.basename(p)
                    if b != bfile and not b.startswith(keep) and b != "books_report.txt":
                        os.remove(p)
                        sent.pop(p, None)
                        with lock:
                            try:
                                remove(f"{R}/new21/{b}")
                            except Exception as e:
                                print(f"[{b} not removed from Drive: {type(e).__name__}: {str(e)[:100]}]", flush=True)
                        print("removed the earlier making of your books:", b, flush=True)
                sh(sys.executable, "new_data.py", "--source", "books", "--file", bfile, "--local", os.path.join(NEW, bfile),
                   "--tok", SRC, "--out", NEW, "--block", *held, check=False)
        for name, tokens, _, flags in SOURCES:
            left = data_end - time.time()
            if left < 120:
                if not started:
                    print(f"no time left to make data: {name} and the rest wait for the next run", flush=True)
                break
            sh(sys.executable, "new_data.py", "--source", name, "--all", "--source_tokens", tokens, "--tok", SRC,
               "--out", NEW, "--block", *held, *flags, timeout=left, check=False)
        shutil.rmtree(os.path.join(NEW, "cache"), ignore_errors=True)

        # ---- training ------------------------------------------------------------------------------------
        extra, have = [], {}
        for name, _, share, _ in SOURCES + [("books", 0, 0, [])]:
            shards = [p for p in glob.glob(os.path.join(NEW, f"{name}_*.bin")) if os.path.exists(p[:-4] + ".json")]
            n = sum(json.load(open(p[:-4] + ".json")).get("tokens", 0) for p in shards)
            if not n:
                continue
            # no source read more than about three times in this round (about 1.5 billion tokens): a small one
            # (your books, YouTube's words) gets a smaller share than planned
            share = min(share or 0.04, 3 * n / 1.5e9)
            have[name] = n
            extra.append(f"{name}={os.path.join(NEW, name)}_*.bin:{share:.4f}")
        print("new data:", ", ".join(f"{k} {v / 1e6:,.0f}M" for k, v in have.items()),
              f"(in all {sum(have.values()) / 1e6:,.0f}M tokens)", flush=True)
        hours = HOURS - (time.time() - T0) / 3600
        assert hours > 1, "less than an hour left for training: run the cell again (the data made is kept in Drive)"
        import torch
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
        batch = "8" if "T4" in gpu or "P100" in gpu else "32"    # rows per GPU: a T4 has 16 GB (weights, optimizer ~8 GB)
        print(f"GPUs: {torch.cuda.device_count()} x {gpu}; {batch} rows per GPU", flush=True)
        sh(sys.executable, "knowledge.py", "--from", SRC, "--data", DATA, "--out", OUT, "--name", NAME,
           "--hours", f"{hours:.2f}", "--tokens", "3e9", "--cooldown", "0.2", "--lr", "1.2e-4", "--batch", batch,
           "--facts_share", "0.10", "--grow", "3", "--grow_memory", "2", "--save_minutes", "120",
           "--light_copy", LIGHT, "--cool_mix", COOL, "--base_mix", BASE, "--extra", *extra)
    finally:
        stop.set()
        thread.join()
        send_new()                                       # the last of it, whatever happened
    push(os.path.join(OUT, NAME + ".zip"), f"{R}/{NAME}.zip")
    print(f"\nsaved to Drive: {R}/{NAME}.zip ({(time.time() - T0) / 3600:.1f} hours). In Drive, harraf/v21_light.pt "
          "(the save to go on from) can be deleted now; harraf/new21 is the new data (keep it for version 22).", flush=True)


_harraf_v21(fs)

from Drive: harraf/harraf2_kaggle_code.zip (0 MB)
code: from Drive (GitHub: HTTPError)
from Drive: harraf/harraf2_base_n.zip (706 MB)
starting from: /tmp/harraf21/model/harraf2_base_n
from Drive: harraf/harraf2/data/meta.json (0 MB)
from Drive: harraf/harraf2/data/morph.json (2 MB)
from Drive: harraf/harraf2/data/tokenizer.json (1 MB)
from Drive: harraf/harraf2/data/train.bin (2,493 MB)
from Drive: harraf/harraf2/data/val.txt (0 MB)
from Drive: harraf/harraf2/data/val_dict.txt (0 MB)
from Drive: harraf/harraf2/data/val_forum.txt (0 MB)
from Drive: harraf/harraf2/data/val_poetry.txt (0 MB)
from Drive: harraf/harraf2/data/val_qa.txt (0 MB)
from Drive: harraf/harraf2/data/val_stories.txt (0 MB)
from Drive: harraf/harraf2/data/val_subs.txt (0 MB)
from Drive: harraf/harraf2/data/val_tweets.txt (0 MB)
from Drive: harraf/harraf2/data/val_web.txt (0 MB)
from Drive: harraf/new21/blocked.npy (0 MB)
from Drive: harraf/new21/blog_dataset_info.bin (0 MB)
from Drive: harraf/new21/blog_dataset_info.j